# Riot API Match Collection

This notebook collects Ranked Solo/Duo match and timeline data from the Riot Games API for later feature engineering and win-condition analysis.

# 1. API Setup

In [ ]:
import json
import os
import random
import time

import requests
from dotenv import load_dotenv

load_dotenv()
RIOT_API_KEY = os.getenv("RIOT_API_KEY")

if not RIOT_API_KEY:
    raise RuntimeError("RIOT_API_KEY is not set. Add it to .env before running this notebook.")

headers = {"X-Riot-Token": RIOT_API_KEY}
print(f"API key loaded: {RIOT_API_KEY is not None}")

# 2. Configuration

In [ ]:
ROUTING = "americas"
PLATFORM = "na1"

TARGET_PATCH = "16.18"
TARGET_QUEUE = 420
RANKED_QUEUE = "RANKED_SOLO_5x5"

TARGET_N = 100
NUM_MASTER_PLAYERS = 20
MATCHES_PER_PLAYER = 50

# 3. Riot API Helpers

The collection flow is Master League-V4 leaderboard → PUUID → Match-V5.

In [ ]:
def riot_get(url, params=None):
    while True:
        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        if response.status_code == 429:
            wait_time = int(
                response.headers.get("Retry-After", 10)
            )

            print(
                f"Rate limited. Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time + 1)
            continue

        response.raise_for_status()
        return response.json()

In [ ]:
def get_match_ids(puuid, count=20):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/by-puuid/{puuid}/ids"
    )

    params = {
        "queue": TARGET_QUEUE,
        "start": 0,
        "count": count
    }

    return riot_get(url, params=params)


def get_match(match_id):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}"
    )

    return riot_get(url)


def get_timeline(match_id):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}/timeline"
    )

    return riot_get(url)


def get_patch(match_data):
    version = match_data["info"]["gameVersion"]
    parts = version.split(".")
    return f"{parts[0]}.{parts[1]}"


def get_random_master_puuids(n=NUM_MASTER_PLAYERS):
    url = (
        f"https://{PLATFORM}.api.riotgames.com/"
        f"lol/league/v4/masterleagues/by-queue/{RANKED_QUEUE}"
    )

    data = riot_get(url)
    entries = data["entries"]

    random.shuffle(entries)

    puuids = []

    for entry in entries:
        puuid = entry.get("puuid")

        if puuid:
            puuids.append(puuid)

        if len(puuids) >= n:
            break

    return puuids

# 4. Sample Master-Tier Players

The source pool is randomly sampled from the NA Master-tier Ranked Solo/Duo league. No personal account data is used.

In [ ]:
master_puuids = get_random_master_puuids()

if not master_puuids:
    raise RuntimeError("No Master-tier players were sampled. Check the API key and platform routing.")

print(f"Master players sampled: {len(master_puuids)}")

# 5. Gather Candidate Match IDs

Match-V5 is first restricted to Ranked Solo/Duo; duplicate match IDs across sampled players are removed.

In [ ]:
candidate_matches = set()

for index, puuid in enumerate(master_puuids, start=1):
    try:
        match_ids = get_match_ids(puuid,count=MATCHES_PER_PLAYER)
        candidate_matches.update(match_ids)
        print(f"Player {index}/{len(master_puuids)}: {len(match_ids)} match IDs")
    except requests.HTTPError as error:
        print(f"Skipping player {index}: {error}")

    time.sleep(0.1)

candidate_matches = list(candidate_matches)
random.shuffle(candidate_matches)
print(f"Unique candidate matches: {len(candidate_matches)}")

In [ ]:
selected_matches = []
selected_match_data = {}

for match_id in candidate_matches:
    if len(selected_matches) >= TARGET_N:
        break

    try:
        match_data = get_match(match_id)
    except requests.HTTPError as error:
        print("Skipping:", match_id, error)
        continue

    if get_patch(match_data) == TARGET_PATCH:
        selected_matches.append(match_id)
        selected_match_data[match_id] = match_data

        print(
            f"{len(selected_matches)}/{TARGET_N}",
            match_id,
            get_patch(match_data)
        )

    time.sleep(0.1)

print("\nSelected matches:", len(selected_matches))

# 6. Filter, Download, and Save Data

Each candidate is checked against the target patch and queue before its match and timeline payloads are saved as a pair under `data/raw/`.

In [ ]:
MATCH_OUTPUT_DIR = "data/raw/matches"
TIMELINE_OUTPUT_DIR = "data/raw/timelines"

os.makedirs(MATCH_OUTPUT_DIR, exist_ok=True)
os.makedirs(TIMELINE_OUTPUT_DIR, exist_ok=True)

for match_id in selected_matches:
    match_data = selected_match_data[match_id]
    match_path = os.path.join(MATCH_OUTPUT_DIR, f"{match_id}.json")
    timeline_path = os.path.join(TIMELINE_OUTPUT_DIR, f"{match_id}.json")

    if os.path.exists(match_path):
        print(f"Match already exists, skipping: {match_id}")
    else:
        with open(match_path, "w") as file:
            json.dump(match_data, file)

    if os.path.exists(timeline_path):
        print(f"Timeline already exists, skipping: {match_id}")
        continue

    try:
        timeline_data = get_timeline(match_id)
    except requests.HTTPError as error:
        print(f"Skipping timeline for {match_id}: {error}")
        continue

    with open(timeline_path, "w") as file:
        json.dump(timeline_data, file)

    print(f"Saved match and timeline: {match_id}")
    time.sleep(0.1)

print(f"Completed collection: {len(selected_matches)} selected matches processed.")

# 7. Inspect a Saved Match

Use this optional check to confirm the saved match metadata, participant count, and timeline structure.

In [ ]:
if not selected_matches:
    raise RuntimeError("No matches were saved, so there is nothing to inspect.")

example_match_id = selected_matches[0]

with open(f"{MATCH_OUTPUT_DIR}/{example_match_id}.json") as file:
    example_match = json.load(file)

with open(f"{TIMELINE_OUTPUT_DIR}/{example_match_id}.json") as file:
    example_timeline = json.load(file)

match_info = example_match["info"]
print(f"Match ID: {example_match['metadata']['matchId']}")
print(f"Queue ID: {match_info['queueId']}")
print(f"Patch: {get_patch(example_match)}")
print(f"Duration: {match_info['gameDuration'] / 60:.1f} minutes")
print(f"Participants: {len(match_info['participants'])}")
print(f"Timeline frames: {len(example_timeline['info']['frames'])}")

In [ ]:
for player in match_info["participants"]:
    print(
        f"{player['teamPosition'] or 'UNKNOWN':<7} "
        f"{player['championName']:<16} "
        f"K/D/A: {player['kills']}/{player['deaths']}/{player['assists']} "
        f"Win: {player['win']}"
    )

print("First five frame timestamps:", [frame["timestamp"] for frame in example_timeline["info"]["frames"][:5]])

# 8. Collection Summary

In [ ]:
print("Patch:", TARGET_PATCH)
print("Queue:", TARGET_QUEUE)
print("Master players sampled:", len(master_puuids))
print("Candidate matches:", len(candidate_matches))
print("Selected matches:", len(selected_matches))